<a href="https://colab.research.google.com/github/cod3astro/kaggle_ML_competition/blob/main/kaggle_podcast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('train.csv', index_col='id')
df.head()

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
for col in df.columns:
  unique_values = df[col].unique()
  print(f"{col} {len(unique_values)} (unique):")
  print(unique_values)

In [ ]:
df['Episode_Title'] = df['Episode_Title'].str.extract(r'(\d+)').astype(int)

In [ ]:
df_drop = df.dropna(axis=0)
df.drop_duplicates(inplace=True)
df_drop.describe()

In [ ]:
df_drop.head()

In [ ]:
df_fill = df.copy()

for col in df_fill.columns:
    if df_fill[col].dtype != 'object':
        df_fill[col] = df_fill[col].fillna(df_fill[col].mean())

print("Original df shape:", df.shape)
print("df_drop shape (NaNs dropped):", df_drop.shape)
print("df_fill shape (NaNs filled):", df_fill.shape)

In [ ]:
df_fill.describe()

In [ ]:
df_fill.head()

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score, mean_squared_error, r2_score

In [ ]:
feature = ['Podcast_Name', 'Episode_Title', 'Episode_Length_minutes', 'Genre','Host_Popularity_percentage',
           'Publication_Day', 'Publication_Time', 'Guest_Popularity_percentage', 'Number_of_Ads', 'Episode_Sentiment']
target = 'Listening_Time_minutes'

X = df_fill[feature]
y = df_fill[target]
A = df_drop[feature]
b = df_drop[target]
train_X, val_X, train_y, val_y = train_test_split(X, y, test_size=0.2, random_state=42)
train_A, val_A, train_b, val_b = train_test_split(A, b, test_size=0.2, random_state=42)

In [ ]:
categorical_col = ['Podcast_Name', 'Genre', 'Publication_Day', 'Publication_Time', 'Episode_Sentiment']
numerical_col = ['Episode_Title', 'Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Number_of_Ads']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_col),
        ('num', 'passthrough', numerical_col)
    ])
model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5))
    ])
model.fit(train_X, train_y)
y_preds = model.predict(val_X)
print('MSE:', mean_squared_error(val_y, y_preds))
print('r2:', r2_score(val_y, y_preds))

In [ ]:
categorical_col = ['Podcast_Name', 'Genre', 'Publication_Day', 'Publication_Time', 'Episode_Sentiment']
numerical_col = ['Episode_Title', 'Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Number_of_Ads']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_col),
        ('num', 'passthrough', numerical_col)
    ])
model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5))
    ])
model.fit(train_A, train_b)
b_preds = model.predict(val_A)
print('MSE:', mean_squared_error(val_b, b_preds))
print('r2:', r2_score(val_b, b_preds))

In [ ]:
test_data = pd.read_csv('test.csv', index_col='id')
#print(test_data.head())
test_data['Episode_Title'] = test_data['Episode_Title'].str.extract(r'(\d+)').astype(int)
test_data = test_data.dropna(axis=0)
test_data.drop_duplicates()
X = test_data[feature]
y_preds = model.predict(X)

submission = pd.DataFrame({
    'ID': test_data.index,
    'Listening_Time_minutes': y_preds
})
submission.to_csv('podcast_submission.csv', index=False)

In [ ]:
from google.colab import files
files.download('podcast_submission.csv')